# Hash Ensemble v2 — KWS-12, with proper diversity controls and accounting

Follow-up to `hash_ensemble_kws.ipynb`. Acts on the v1 critique:

1. **Diversity-source controls (C1).** v1 conflated hash, init, and data-order seeds — couldn't attribute the ensemble gain to its actual source. v2 separates them: `hash-only-different`, `init-only-different`, `all-different` arms.
2. **N-sweep (C2).** Pool of 6 trained models, accuracy and std as functions of N=1..6. Tests whether the v1 ×3.84 std-reduction at N=3 is real or saturation.
3. **Oracle and disagreement (C3).** Upper bound (any-correct vote) and pairwise disagreement — quantifies remaining ensemble headroom.
4. **Strict per-node vs fleet accounting.** Every result reports per-node and total real/codebook params explicitly.
5. **Dropped:** branches (refuted on KWS) and joint training (no win even with hybrid loss).

Defaults: full ~2.5h on T4, FAST_MODE ~35-45 min.

In [ ]:
try:
    from torchaudio.datasets import SPEECHCOMMANDS
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torchaudio'])
    from torchaudio.datasets import SPEECHCOMMANDS

In [ ]:
import os
from pathlib import Path
from torchaudio.datasets import SPEECHCOMMANDS

DATA_ROOT = Path(os.environ.get('SPEECHCOMMANDS_DATA_ROOT', './data'))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
for sub in ('training', 'validation', 'testing'):
    SPEECHCOMMANDS(root=str(DATA_ROOT), download=True, subset=sub)
print(f'Speech Commands at {DATA_ROOT.resolve()}')

## Imports + config

Same architecture as v1 (so feature cache from v1 is reused if present).

In [ ]:
import math, time, json, random, hashlib
from collections import Counter
from typing import Any
from tqdm.auto import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchaudio
import matplotlib.pyplot as plt

FAST_MODE = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESULTS_PATH = './results_kws_v2.json'

COMMANDS = ['yes','no','up','down','left','right','on','off','stop','go']
ALL_LABELS = COMMANDS + ['unknown', 'silence']
LABEL_TO_IDX = {l: i for i, l in enumerate(ALL_LABELS)}
NUM_LABELS = len(ALL_LABELS)

SAMPLE_RATE = 16_000; CLIP_SAMPLES = 16_000
WINDOW_MS = 30; HOP_MS = 20; N_MELS = 40
WINDOW_SAMPLES = (SAMPLE_RATE * WINDOW_MS) // 1000
HOP_SAMPLES = (SAMPLE_RATE * HOP_MS) // 1000
FRAME_COUNT = 1 + max(0, (CLIP_SAMPLES - WINDOW_SAMPLES) // HOP_SAMPLES)
LOG_OFFSET = 1e-6

CHANNELS = 64
NUM_BLOCKS = 4
USE_RESIDUAL = True
SIGNED_HASH = True
HASH_AWARE_INIT = True
DROPOUT = 0.0

ENSEMBLE_N = 3
K_HOMO = 500
K_HETERO = [256, 500, 768]
K_SWEEP = [256, 500, 1024] if FAST_MODE else [128, 256, 500, 1024]

BATCH_SIZE = 128
LR = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.02
GRAD_CLIP = 1.0
EPOCHS = 8 if FAST_MODE else 20

POOL_SIZE = 4 if FAST_MODE else 6                  # for N-sweep
CONTROL_SEEDS_PER_ARM = 2 if FAST_MODE else 3      # for C1 control ablations

CACHE_ROOT = DATA_ROOT / 'feature_cache'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}  FAST: {FAST_MODE}  EPOCHS={EPOCHS}  POOL_SIZE={POOL_SIZE}')
print(f'K_HOMO={K_HOMO}  K_HETERO={K_HETERO}  K_SWEEP={K_SWEEP}')

def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_all_seeds(13)

results = {}
def save_results():
    with open(RESULTS_PATH, 'w') as f:
        json.dump(results, f, indent=2)

## Feature cache (re-uses v1 cache if signature matches)

In [ ]:
_MEL = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=WINDOW_SAMPLES, win_length=WINDOW_SAMPLES,
    hop_length=HOP_SAMPLES, n_mels=N_MELS, center=False, power=2.0,
)

def waveform_to_logmel(wav):
    spec = _MEL(wav.to(torch.float32))
    spec = torch.log(spec + LOG_OFFSET)
    return (spec - spec.mean()) / (spec.std() + 1e-5)

def cache_signature(subset):
    payload = {'version': 'kws_v1', 'subset': subset, 'commands': COMMANDS,
               'sr': SAMPLE_RATE, 'clip': CLIP_SAMPLES, 'win_ms': WINDOW_MS,
               'hop_ms': HOP_MS, 'n_mels': N_MELS, 'normalize': 'instance',
               'frontend': 'log_mel', 'unknown_fraction': 1.0,
               'silence_fraction': 0.12, 'silence_ref': 'known'}
    return hashlib.sha1(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:16]


class CachedSpeechCommands(Dataset):
    def __init__(self, subset):
        self.subset = subset
        self.base = SPEECHCOMMANDS(root=str(DATA_ROOT), download=False, subset=subset)
        self.base_path = Path(getattr(self.base, '_path', DATA_ROOT))
        self.cache_path = CACHE_ROOT / f'{cache_signature(subset)}.pt'
        walker = list(getattr(self.base, '_walker', []))
        rng = random.Random(13 + {'training':0,'validation':1,'testing':2}[subset])
        wanted = set(COMMANDS); known, unknown = [], []
        for i, item in enumerate(walker):
            label = self._label_from_item(i, item)
            if label in wanted: known.append(i)
            elif label != '_background_noise_': unknown.append(i)
        rng.shuffle(unknown)
        sel_unknown = unknown[:min(len(unknown), len(known))]
        sil_target = max(1, int(round(len(known) * 0.12)))
        self.entries = []
        for i in known:
            self.entries.append(('speech', i, LABEL_TO_IDX[self._label_from_item(i, walker[i])]))
        for i in sel_unknown:
            self.entries.append(('speech', i, LABEL_TO_IDX['unknown']))
        for s in range(sil_target):
            self.entries.append(('silence', s, LABEL_TO_IDX['silence']))
        rng.shuffle(self.entries)
        self.background_noises = self._load_bg()
        self.cached_features = None; self.cached_labels = None

    def _label_from_item(self, i, item):
        if isinstance(item, str):
            p = Path(item)
            if p.parent.name: return p.parent.name
        _, _, label, *_ = self.base[i]
        return str(label)

    def _load_bg(self):
        nd = None
        for d in [self.base_path/'_background_noise_', self.base_path.parent/'_background_noise_']:
            if d.exists(): nd = d; break
        if nd is None: return []
        out = []
        for p in sorted(nd.glob('*.wav')):
            w, sr = torchaudio.load(str(p)); w = w.mean(0, keepdim=True)
            if sr != SAMPLE_RATE: w = torchaudio.functional.resample(w, sr, SAMPLE_RATE)
            out.append(w)
        return out

    def _prep(self, w, sr):
        w = w.mean(0, keepdim=True)
        if sr != SAMPLE_RATE: w = torchaudio.functional.resample(w, sr, SAMPLE_RATE)
        if w.shape[1] < CLIP_SAMPLES: w = F.pad(w, (0, CLIP_SAMPLES - w.shape[1]))
        else: w = w[:, :CLIP_SAMPLES]
        return w

    def _silence(self, idx):
        if not self.background_noises: return torch.zeros(1, CLIP_SAMPLES)
        n = self.background_noises[idx % len(self.background_noises)]
        if n.shape[1] <= CLIP_SAMPLES:
            return F.pad(n, (0, max(CLIP_SAMPLES - n.shape[1], 0)))[:, :CLIP_SAMPLES] * 0.1
        s = (idx * 9973) % (n.shape[1] - CLIP_SAMPLES + 1)
        return n[:, s:s+CLIP_SAMPLES] * 0.1

    def _compute(self, idx):
        kind, base_idx, lid = self.entries[idx]
        if kind == 'speech':
            w, sr, *_ = self.base[int(base_idx)]
            w = self._prep(w, int(sr))
        else:
            w = self._silence(idx)
        return waveform_to_logmel(w).squeeze(0), lid

    def materialize(self):
        if self.cached_features is not None:
            return {'status': 'memory', 'items': len(self.entries)}
        if self.cache_path.exists():
            p = torch.load(self.cache_path, map_location='cpu')
            self.cached_features = p['features']; self.cached_labels = p['labels']
            return {'status': 'loaded', 'items': int(self.cached_labels.numel()), 'path': str(self.cache_path)}
        feats, labels = [], []
        for i in tqdm(range(len(self.entries)), desc=f'cache {self.subset}', leave=False):
            f, lid = self._compute(i)
            feats.append(f.to(torch.float32)); labels.append(int(lid))
        self.cached_features = torch.stack(feats, 0)
        self.cached_labels = torch.tensor(labels, dtype=torch.int64)
        torch.save({'features': self.cached_features, 'labels': self.cached_labels}, self.cache_path)
        return {'status': 'built', 'items': int(self.cached_labels.numel()), 'path': str(self.cache_path)}

    def __len__(self): return len(self.entries)
    def __getitem__(self, idx):
        if self.cached_features is not None:
            return self.cached_features[idx], int(self.cached_labels[idx].item())
        return self._compute(idx)

trainset = CachedSpeechCommands('training')
valset   = CachedSpeechCommands('validation')
testset  = CachedSpeechCommands('testing')
for n, ds in [('train', trainset), ('val', valset), ('test', testset)]:
    print(n, ds.materialize())
valloader  = DataLoader(valset,  batch_size=256, shuffle=False, num_workers=0, pin_memory=True)
testloader = DataLoader(testset, batch_size=256, shuffle=False, num_workers=0, pin_memory=True)

## Architecture (same as v1)

In [ ]:
P_OC, P_IC, P_KH, P_KW, P_LAYER = 1337, 7919, 2971, 6151, 104729
P_SEED = 31337
S_A, S_B, S_C, S_SEED = 4099, 6151, 14887, 67867

def _bits_to_sign(b): return b.to(torch.float32).mul_(2.0).sub_(1.0)
def _aware_std(fan_in_eq): return 1.0 / math.sqrt(max(fan_in_eq, 1))


class HashedLinear(nn.Module):
    def __init__(self, in_dim, out_dim, K, layer_id, hash_seed=0, signed=True, aware_init=True):
        super().__init__()
        self.K = K; self.signed = signed
        std = _aware_std(in_dim) if aware_init else 0.01
        self.codebook = nn.Parameter(torch.randn(K) * std)
        self.bias = nn.Parameter(torch.zeros(out_dim))
        i = torch.arange(out_dim).view(-1, 1); j = torch.arange(in_dim).view(1, -1)
        h = (i*P_OC + j*P_IC + layer_id*P_LAYER + hash_seed*P_SEED) % K
        s = _bits_to_sign((i*S_A + j*S_B + layer_id*S_C + hash_seed*S_SEED) % 2)
        self.register_buffer('hash_idx', h, persistent=False)
        self.register_buffer('signs', s, persistent=False)
    def materialize(self):
        W = self.codebook[self.hash_idx]
        return W * self.signs if self.signed else W
    def forward(self, x): return F.linear(x, self.materialize(), self.bias)
    def virtual_params(self): return self.hash_idx.numel()
    def real_params(self): return self.K + self.bias.numel()

class HashedConv2d(nn.Module):
    def __init__(self, ic, oc, k, K, layer_id, hash_seed=0, stride=1, padding=0, signed=True, aware_init=True):
        super().__init__()
        self.K = K; self.signed = signed; self.stride = stride; self.padding = padding
        kh, kw = (k, k) if isinstance(k, int) else k
        std = _aware_std(ic*kh*kw) if aware_init else 0.01
        self.codebook = nn.Parameter(torch.randn(K) * std)
        self.bias = nn.Parameter(torch.zeros(oc))
        a = torch.arange(oc).view(-1,1,1,1); b = torch.arange(ic).view(1,-1,1,1)
        c = torch.arange(kh).view(1,1,-1,1); d = torch.arange(kw).view(1,1,1,-1)
        h = (a*P_OC + b*P_IC + c*P_KH + d*P_KW + layer_id*P_LAYER + hash_seed*P_SEED) % K
        s = _bits_to_sign((a*S_A + b*S_B + c*S_C + d*(S_A+S_B) + layer_id*(S_C+11) + hash_seed*S_SEED) % 2)
        self.register_buffer('hash_idx', h, persistent=False)
        self.register_buffer('signs', s, persistent=False)
    def materialize(self):
        W = self.codebook[self.hash_idx]
        return W * self.signs if self.signed else W
    def forward(self, x):
        return F.conv2d(x, self.materialize(), self.bias, stride=self.stride, padding=self.padding)
    def virtual_params(self): return self.hash_idx.numel()
    def real_params(self): return self.K + self.bias.numel()

class HashedDepthwiseConv2d(nn.Module):
    def __init__(self, ch, k, K, layer_id, hash_seed=0, stride=1, padding=0, signed=True, aware_init=True):
        super().__init__()
        self.K = K; self.signed = signed; self.channels = ch
        self.stride = stride; self.padding = padding
        kh, kw = (k, k) if isinstance(k, int) else k
        std = _aware_std(kh*kw) if aware_init else 0.01
        self.codebook = nn.Parameter(torch.randn(K) * std)
        self.bias = nn.Parameter(torch.zeros(ch))
        a = torch.arange(ch).view(-1,1,1); b = torch.arange(kh).view(1,-1,1); c = torch.arange(kw).view(1,1,-1)
        h = (a*P_OC + b*P_KH + c*P_KW + layer_id*P_LAYER + hash_seed*P_SEED) % K
        s = _bits_to_sign((a*S_A + b*S_B + c*S_C + layer_id*(S_A+29) + hash_seed*S_SEED) % 2)
        self.register_buffer('hash_idx', h, persistent=False)
        self.register_buffer('signs', s, persistent=False)
    def materialize(self):
        W = self.codebook[self.hash_idx]
        if self.signed: W = W * self.signs
        return W.unsqueeze(1)
    def forward(self, x):
        return F.conv2d(x, self.materialize(), self.bias, stride=self.stride,
                        padding=self.padding, groups=self.channels)
    def virtual_params(self): return self.hash_idx.numel()
    def real_params(self): return self.K + self.bias.numel()

HASHED_LAYER_TYPES = (HashedLinear, HashedConv2d, HashedDepthwiseConv2d)

class DSCNNBlock(nn.Module):
    def __init__(self, ch, dw, pw, residual=True):
        super().__init__()
        self.dw = dw; self.bn_dw = nn.BatchNorm2d(ch)
        self.pw = pw; self.bn_pw = nn.BatchNorm2d(ch); self.residual = residual
    def forward(self, x):
        s = x
        x = F.relu(self.bn_dw(self.dw(x)))
        x = self.bn_pw(self.pw(x))
        if self.residual: x = x + s
        return F.relu(x)

class HashedKWS(nn.Module):
    def __init__(self, K_per_layer, hash_seed=0, channels=CHANNELS, num_blocks=NUM_BLOCKS,
                 num_classes=NUM_LABELS, residual=USE_RESIDUAL, signed=SIGNED_HASH,
                 aware_init=HASH_AWARE_INIT, dropout=DROPOUT):
        super().__init__()
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        if isinstance(K_per_layer, int):
            K_stem = K_per_layer; K_dw = [K_per_layer]*num_blocks
            K_pw = [K_per_layer]*num_blocks; K_fc = K_per_layer
        else:
            K_stem = K_per_layer.get('stem', None)
            K_dw = K_per_layer.get('dw', [None]*num_blocks)
            K_pw = K_per_layer.get('pw', [None]*num_blocks)
            K_fc = K_per_layer.get('fc', None)
            if isinstance(K_dw, int): K_dw = [K_dw]*num_blocks
            if isinstance(K_pw, int): K_pw = [K_pw]*num_blocks
        kwh = dict(signed=signed, aware_init=aware_init)
        layer_id = 0
        if K_stem is None:
            self.stem = nn.Conv2d(1, channels, 3, stride=2, padding=1)
        else:
            self.stem = HashedConv2d(1, channels, 3, K_stem, layer_id, hash_seed,
                                     stride=2, padding=1, **kwh); layer_id += 1
        self.bn_stem = nn.BatchNorm2d(channels)
        blocks = []
        for b in range(num_blocks):
            kdw = K_dw[b] if b < len(K_dw) else K_dw[-1]
            kpw = K_pw[b] if b < len(K_pw) else K_pw[-1]
            if kdw is None:
                dw = nn.Conv2d(channels, channels, 3, padding=1, groups=channels)
            else:
                dw = HashedDepthwiseConv2d(channels, 3, kdw, layer_id, hash_seed, padding=1, **kwh)
                layer_id += 1
            if kpw is None:
                pw = nn.Conv2d(channels, channels, 1)
            else:
                pw = HashedConv2d(channels, channels, 1, kpw, layer_id, hash_seed, **kwh)
                layer_id += 1
            blocks.append(DSCNNBlock(channels, dw, pw, residual=residual))
        self.blocks = nn.ModuleList(blocks)
        self.pool = nn.AdaptiveAvgPool2d(1)
        if K_fc is None:
            self.fc = nn.Linear(channels, num_classes)
        else:
            self.fc = HashedLinear(channels, num_classes, K_fc, layer_id, hash_seed, **kwh)

    def forward(self, x):
        if x.dim() == 3: x = x.unsqueeze(1)
        x = F.relu(self.bn_stem(self.stem(x)))
        for blk in self.blocks: x = blk(x)
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(self.dropout(x))

    def real_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)
    def codebook_params(self):
        return sum(m.K for m in self.modules() if isinstance(m, HASHED_LAYER_TYPES))
    def virtual_params(self):
        n = 0
        for m in self.modules():
            if m is self: continue
            if isinstance(m, HASHED_LAYER_TYPES): n += m.virtual_params()
            elif isinstance(m, (nn.Conv2d, nn.Linear)): n += m.weight.numel()
        return n

_m = HashedKWS(K_per_layer=K_HOMO)
print(f'sanity K={K_HOMO}: real={_m.real_params():,}  '
      f'codebook={_m.codebook_params():,}  virtual={_m.virtual_params():,}')
del _m

## Training with explicit (init_seed, hash_seed, loader_seed) control

v2 separates three randomness sources. `train_with_seeds(init_seed, hash_seed, loader_seed, K)` constructs a model and a DataLoader with independent seeds, so the C1 control ablations can isolate each source's contribution.

In [ ]:
def make_train_loader(seed, batch_size=BATCH_SIZE):
    g = torch.Generator(); g.manual_seed(int(seed))
    return DataLoader(trainset, batch_size=batch_size, shuffle=True,
                       generator=g, num_workers=0, pin_memory=True)

def build_model(init_seed, hash_seed, K_per_layer, **kwargs):
    set_all_seeds(int(init_seed))
    return HashedKWS(K_per_layer=K_per_layer, hash_seed=int(hash_seed), **kwargs)

@torch.no_grad()
def evaluate_logits(model, loader):
    model.eval()
    al, ay = [], []
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        lg = model(x)
        correct += (lg.argmax(-1) == y).sum().item(); total += y.size(0)
        al.append(lg.cpu()); ay.append(y.cpu())
    return correct / total, torch.cat(al), torch.cat(ay)

def train_one_model(model, loader, epochs=EPOCHS, log_prefix=''):
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_val, best_state = 0.0, None
    t0 = time.time()
    for ep in range(epochs):
        model.train(); ema = None
        pbar = tqdm(loader, desc=f'{log_prefix} ep{ep+1}/{epochs}', leave=False)
        for x, y in pbar:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = F.cross_entropy(model(x), y, label_smoothing=LABEL_SMOOTHING)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            ema = loss.item() if ema is None else 0.9 * ema + 0.1 * loss.item()
            pbar.set_postfix(loss=f'{ema:.3f}', lr=f'{sched.get_last_lr()[0]:.4f}')
        sched.step()
        val_acc, _, _ = evaluate_logits(model, valloader)
        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        tqdm.write(f'{log_prefix} ep{ep+1}/{epochs}  val={val_acc:.4f}  '
                   f'best={best_val:.4f}  ({time.time()-t0:.0f}s)')
    if best_state is not None:
        model.load_state_dict(best_state)
    test_acc, test_logits, test_labels = evaluate_logits(model, testloader)
    return model, test_acc, best_val, test_logits, test_labels

def train_with_seeds(init_seed, hash_seed, loader_seed, K, log_prefix='', epochs=EPOCHS):
    m = build_model(init_seed, hash_seed, K_per_layer=K)
    loader = make_train_loader(loader_seed)
    return train_one_model(m, loader, epochs=epochs, log_prefix=log_prefix)

def param_accounting(models):
    """Returns per-node and fleet aggregates."""
    per_node_real = [m.real_params() for m in models]
    per_node_cb   = [m.codebook_params() for m in models]
    return {
        'per_node_real': per_node_real,
        'per_node_codebook': per_node_cb,
        'total_real': sum(per_node_real),
        'total_codebook': sum(per_node_cb),
        'virtual_per_node': models[0].virtual_params() if models else 0,
    }

## Aggregators (incl. oracle)

Oracle = upper bound. `oracle_correct(per_model_logits, labels)` returns 1 if any member gets it right. Tells us the absolute headroom for any router.

In [ ]:
def agg_mean_logits(L): return L.mean(0)
def agg_mean_probs(L):  return F.softmax(L, dim=-1).mean(0)
def agg_majority_vote(L):
    preds = L.argmax(-1)
    return F.one_hot(preds, num_classes=L.size(-1)).float().mean(0)

AGGREGATORS = {
    'mean_logits': agg_mean_logits,
    'mean_probs': agg_mean_probs,
    'majority_vote': agg_majority_vote,
}

def ensemble_accuracy(per_model_logits, labels, agg):
    out = agg(torch.stack(per_model_logits))
    return (out.argmax(-1) == labels).float().mean().item()

def oracle_accuracy(per_model_logits, labels):
    preds = torch.stack([lg.argmax(-1) for lg in per_model_logits])  # [N, B]
    any_correct = (preds == labels.unsqueeze(0)).any(0)
    return any_correct.float().mean().item()

def disagreement_matrix(per_model_logits):
    preds = [lg.argmax(-1) for lg in per_model_logits]
    N = len(preds); D = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            D[i, j] = (preds[i] != preds[j]).float().mean().item()
    return D

## A0. Sanity baseline at K=K_HOMO

Quick check the architecture still hits ≈0.91 with v2's strict-seed harness.

In [ ]:
_, t0_acc, _, _, _ = train_with_seeds(init_seed=42, hash_seed=42, loader_seed=42,
                                       K=K_HOMO, log_prefix='[A0]')
results['A0'] = {'test_acc': t0_acc, 'K': K_HOMO}
print(f'A0 single K={K_HOMO}: test={t0_acc:.4f}')
save_results()

## A1. K-sweep

Confirms the K → accuracy curve and provides reference points for budget comparisons.

In [ ]:
results['A1_sweep'] = {}
for K in K_SWEEP:
    m, t, v, lg, lbl = train_with_seeds(init_seed=100+K, hash_seed=100+K, loader_seed=100+K,
                                         K=K, log_prefix=f'[A1 K={K}]')
    results['A1_sweep'][K] = {
        'test': t, 'val': v,
        'real': m.real_params(), 'codebook': m.codebook_params(),
        'virtual': m.virtual_params(),
    }
    print(f'A1 K={K}: test={t:.4f}  real={m.real_params():,}  codebook={m.codebook_params():,}')
    del m
save_results()

## C1. Diversity-source controls

Three arms, `CONTROL_SEEDS_PER_ARM` models each, all at K=K_HOMO:

- **hash_only**: shared `init_seed=A`, shared `loader_seed=A`, varied `hash_seed`
- **init_only**: varied `init_seed`, shared `hash_seed=A`, shared `loader_seed=A`
- **all_diff**: varied `init_seed`, varied `hash_seed`, varied `loader_seed`

We report each arm's single-model accs, ensemble acc with `mean_logits`, oracle, mean pairwise disagreement. The `all_diff` arm doubles as the "normal" homogeneous-K ensemble (was A2 in v1).

**Reading:** if `hash_only` ensemble gain ≈ `all_diff` ensemble gain, the gain truly comes from hash diversity — confirming the diploma narrative. If `init_only` gain is comparable, init diversity is the main contributor and the story changes.

In [ ]:
def run_arm(arm, seed_block):
    """Trains CONTROL_SEEDS_PER_ARM models per the arm's seed protocol."""
    accs, logits = [], []
    for k in range(CONTROL_SEEDS_PER_ARM):
        if arm == 'hash_only':
            init_s, loader_s, hash_s = 7777, 7777, seed_block + k
        elif arm == 'init_only':
            init_s, loader_s, hash_s = seed_block + k, 7777, 7777
        elif arm == 'all_diff':
            init_s, loader_s, hash_s = seed_block + k, seed_block + k, seed_block + k
        else:
            raise ValueError(arm)
        m, t, _, lg, lbl = train_with_seeds(init_s, hash_s, loader_s, K=K_HOMO,
                                              log_prefix=f'[C1 {arm} m{k}]')
        accs.append(t); logits.append(lg)
        del m
    return accs, logits, lbl

ARMS = ['hash_only', 'init_only', 'all_diff']
SEED_BLOCKS = {'hash_only': 1000, 'init_only': 2000, 'all_diff': 3000}

C1 = {}
labels_ref = None
for arm in ARMS:
    accs, logits, lbl = run_arm(arm, SEED_BLOCKS[arm])
    if labels_ref is None: labels_ref = lbl
    ens_logits = ensemble_accuracy(logits, lbl, agg_mean_logits)
    ens_probs = ensemble_accuracy(logits, lbl, agg_mean_probs)
    oracle = oracle_accuracy(logits, lbl)
    D = disagreement_matrix(logits)
    mean_disagree = float(D[np.triu_indices(D.shape[0], k=1)].mean())
    C1[arm] = {
        'singles': accs, 'best_single': max(accs), 'mean_single': float(np.mean(accs)),
        'ens_mean_logits': ens_logits, 'ens_mean_probs': ens_probs,
        'oracle': oracle, 'mean_pairwise_disagreement': mean_disagree,
        'gain_over_best': ens_logits - max(accs),
        'gain_over_mean': ens_logits - float(np.mean(accs)),
    }
    # keep all_diff logits for reuse in C2/C3
    if arm == 'all_diff':
        all_diff_logits = logits
    print(f'C1 {arm}: singles={accs}  ens_logits={ens_logits:.4f}  oracle={oracle:.4f}  '
          f'disagree={mean_disagree:.3f}  gain_over_best={ens_logits - max(accs):+.4f}')

results['C1_controls'] = C1
save_results()

## C2. Pool of trained models for N-sweep

Extend the `all_diff` pool to `POOL_SIZE` models (already have `CONTROL_SEEDS_PER_ARM`; train the rest). Then for each N from 1 to `POOL_SIZE`, evaluate ensemble accuracy and std across all subsets of size N.

**Reading:** the curves answer (a) does ensemble accuracy still grow at N=4,5,6 or saturates at N=3? (b) is the std-reduction better than √N (super-bagging) or matches it (plain bagging)?

In [ ]:
from itertools import combinations

pool_logits = list(all_diff_logits)
pool_accs = list(C1['all_diff']['singles'])
next_seed = SEED_BLOCKS['all_diff'] + CONTROL_SEEDS_PER_ARM
while len(pool_logits) < POOL_SIZE:
    k = len(pool_logits)
    s = next_seed + k
    m, t, _, lg, _ = train_with_seeds(s, s, s, K=K_HOMO, log_prefix=f'[C2 pool m{k}]')
    pool_logits.append(lg); pool_accs.append(t); del m

labels = labels_ref
n_sweep = []
for N in range(1, POOL_SIZE + 1):
    accs_n = []
    for idxs in combinations(range(POOL_SIZE), N):
        sub = [pool_logits[i] for i in idxs]
        accs_n.append(ensemble_accuracy(sub, labels, agg_mean_logits))
    n_sweep.append({'N': N, 'mean': float(np.mean(accs_n)), 'std': float(np.std(accs_n)),
                    'count': len(accs_n)})
    print(f'C2 N={N}: mean={np.mean(accs_n):.4f}  std={np.std(accs_n):.5f}  '
          f'subsets={len(accs_n)}')

results['C2_n_sweep'] = {
    'pool_size': POOL_SIZE,
    'pool_singles': pool_accs,
    'curves': n_sweep,
}
save_results()

## C3. Oracle and disagreement diagnostics

Using the full pool. Oracle = upper bound under any router. Per-class disagreement shows which classes are most ambiguous across the ensemble — gives a hint of what a smart router would need to handle.

In [ ]:
ens_acc_pool = ensemble_accuracy(pool_logits, labels, agg_mean_logits)
oracle_acc_pool = oracle_accuracy(pool_logits, labels)
D = disagreement_matrix(pool_logits)
print(f'Pool size={POOL_SIZE}, ensemble (mean_logits) = {ens_acc_pool:.4f}')
print(f'Oracle (any-correct) = {oracle_acc_pool:.4f}')
print(f'Headroom (oracle − ens) = {oracle_acc_pool - ens_acc_pool:+.4f}')
print('Pairwise disagreement matrix:')
with np.printoptions(precision=3, suppress=True): print(D)

# Per-class disagreement: how often do members disagree on each true class?
per_class_disagree = np.zeros(NUM_LABELS)
per_class_count = np.zeros(NUM_LABELS)
preds_stack = torch.stack([lg.argmax(-1) for lg in pool_logits])  # [N, B]
for c in range(NUM_LABELS):
    mask = (labels == c)
    if mask.sum() == 0: continue
    sub = preds_stack[:, mask]  # [N, n_c]
    # disagreement rate per sample: 1 - (max-vote count / N)
    n_samples = sub.size(1)
    rates = []
    for s in range(n_samples):
        votes = sub[:, s].numpy()
        from collections import Counter as _C
        max_count = max(_C(votes).values())
        rates.append(1.0 - max_count / POOL_SIZE)
    per_class_disagree[c] = float(np.mean(rates))
    per_class_count[c] = int(mask.sum().item())

results['C3_oracle_disagreement'] = {
    'ens_pool_mean_logits': ens_acc_pool,
    'oracle_pool': oracle_acc_pool,
    'headroom': oracle_acc_pool - ens_acc_pool,
    'pairwise_disagreement': D.tolist(),
    'per_class_disagreement': {
        ALL_LABELS[c]: {'disagree_rate': float(per_class_disagree[c]),
                         'n_samples': int(per_class_count[c])}
        for c in range(NUM_LABELS)
    },
}
print('\nPer-class disagreement:')
for c in range(NUM_LABELS):
    print(f'  {ALL_LABELS[c]:>10s}: rate={per_class_disagree[c]:.3f}  n={int(per_class_count[c])}')
save_results()

## A3 (legacy). Heterogeneous K ensemble

Kept for cross-task continuity with v1. Three models with K = `K_HETERO`, all-different seeds.

In [ ]:
het_logits, het_singles, het_models = [], [], []
for k, K in enumerate(K_HETERO):
    s = 5000 + k
    m, t, _, lg, lbl = train_with_seeds(s, s, s, K=K, log_prefix=f'[A3 m{k} K={K}]')
    het_logits.append(lg); het_singles.append(t); het_models.append(m)
    print(f'A3 m{k} K={K}: test={t:.4f}')

ens_het = ensemble_accuracy(het_logits, lbl, agg_mean_logits)
results['A3_hetero'] = {
    'K_per_model': K_HETERO,
    'singles': het_singles, 'best_single': max(het_singles),
    'ens_mean_logits': ens_het,
    'oracle': oracle_accuracy(het_logits, lbl),
    'param_accounting': param_accounting(het_models),
}
print(f'A3 hetero ensemble: {ens_het:.4f}  best single: {max(het_singles):.4f}  '
      f'oracle: {results["A3_hetero"]["oracle"]:.4f}')
save_results()
del het_models

## A5 (legacy single). pw_heavy single — deployment-friendly variant

Hash only pointwise layers; stem, depthwise, fc stay dense. Practically simpler on ESP32 (no hashed conv stem/dw kernel needed). On KWS this was on-par with full-hashed in v1; we just confirm at one K and move on.

In [ ]:
pwh_kdict = {'stem': None, 'dw': [None]*NUM_BLOCKS, 'pw': [K_HOMO]*NUM_BLOCKS, 'fc': None}
m_pwh = build_model(init_seed=6000, hash_seed=6000, K_per_layer=pwh_kdict)
loader = make_train_loader(6000)
_, t_pwh, _, _, _ = train_one_model(m_pwh, loader, log_prefix=f'[A5 pwh K_pw={K_HOMO}]')
results['A5_pw_heavy_single'] = {
    'K_per_pw': K_HOMO, 'test_acc': t_pwh,
    'real': m_pwh.real_params(), 'codebook': m_pwh.codebook_params(),
}
print(f'A5 pw_heavy K_pw={K_HOMO}: test={t_pwh:.4f}  real={m_pwh.real_params():,}')
del m_pwh
save_results()

## Param accounting summary (per-node vs fleet)

Honest-budget tables for the diploma.

In [ ]:
# A1 single reference points
print('Single hashed models (A1 sweep):')
for K in sorted(results['A1_sweep'].keys()):
    v = results['A1_sweep'][K]
    print(f'  K={K:>5d}: test={v["test"]:.4f}  per_node_real={v["real"]:,}  '
          f'per_node_codebook={v["codebook"]:,}  virtual={v["virtual"]:,}')

# Pool ensembles
print('\nHomogeneous K=K_HOMO ensembles (from C2 pool):')
single_real = results['A1_sweep'][K_HOMO]['real']
single_cb   = results['A1_sweep'][K_HOMO]['codebook']
for entry in results['C2_n_sweep']['curves']:
    N = entry['N']
    print(f'  N={N}: ens_mean={entry["mean"]:.4f}  std={entry["std"]:.5f}  '
          f'per_node_real={single_real:,}  fleet_real={single_real*N:,}  '
          f'fleet_codebook={single_cb*N:,}')

# A3
print('\nHeterogeneous K ensemble (A3):')
acct = results['A3_hetero']['param_accounting']
print(f'  per_node_real={acct["per_node_real"]}  per_node_codebook={acct["per_node_codebook"]}')
print(f'  fleet_real={acct["total_real"]:,}  fleet_codebook={acct["total_codebook"]:,}  '
      f'virtual={acct["virtual_per_node"]:,}')

save_results()

## Summary

In [ ]:
print('='*78); print('SUMMARY — KWS-12 v2'); print('='*78)
print(f'A0 sanity K={K_HOMO}:                          {results["A0"]["test_acc"]:.4f}')
print('--- A1 sweep (single hashed) ---')
for K in sorted(results['A1_sweep'].keys()):
    v = results['A1_sweep'][K]
    print(f'  K={K:>5d}: test={v["test"]:.4f}  real={v["real"]:,}')
print('--- C1 control ablations (3 arms × {} seeds, K={}) ---'.format(
    CONTROL_SEEDS_PER_ARM, K_HOMO))
for arm in ARMS:
    a = results['C1_controls'][arm]
    print(f'  {arm:>10s}: singles_mean={a["mean_single"]:.4f}  '
          f'ens={a["ens_mean_logits"]:.4f}  oracle={a["oracle"]:.4f}  '
          f'gain_over_mean={a["gain_over_mean"]:+.4f}  disagree={a["mean_pairwise_disagreement"]:.3f}')
print('--- C2 N-sweep (pool size {}, K={}) ---'.format(POOL_SIZE, K_HOMO))
for c in results['C2_n_sweep']['curves']:
    print(f'  N={c["N"]}: mean={c["mean"]:.4f}  std={c["std"]:.5f}  subsets={c["count"]}')
print('--- C3 oracle/disagreement (pool) ---')
v = results['C3_oracle_disagreement']
print(f'  ens (mean_logits): {v["ens_pool_mean_logits"]:.4f}')
print(f'  oracle:            {v["oracle_pool"]:.4f}')
print(f'  headroom:          {v["headroom"]:+.4f}')
print('--- A3 heterogeneous K ---')
v = results['A3_hetero']
print(f'  best single: {v["best_single"]:.4f}  ens: {v["ens_mean_logits"]:.4f}  oracle: {v["oracle"]:.4f}')
print('--- A5 pw_heavy single ---')
v = results['A5_pw_heavy_single']
print(f'  K_pw={v["K_per_pw"]}: test={v["test_acc"]:.4f}  real={v["real"]:,}')
print('='*78)
save_results()

In [ ]:
# Headline plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) C1: gain by diversity source
ax = axes[0]
arms = list(C1.keys())
ens = [C1[a]['ens_mean_logits'] for a in arms]
best = [C1[a]['best_single'] for a in arms]
mean_s = [C1[a]['mean_single'] for a in arms]
x = np.arange(len(arms)); w = 0.28
ax.bar(x - w, mean_s, w, label='single (mean over arm)')
ax.bar(x,     best,   w, label='best single')
ax.bar(x + w, ens,    w, label='ensemble (mean_logits)')
ax.set_xticks(x); ax.set_xticklabels(arms); ax.set_ylabel('test acc')
ax.set_title('C1 — diversity source')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

# (b) C2: N-sweep accuracy and std
ax = axes[1]
ns = [c['N'] for c in results['C2_n_sweep']['curves']]
mns = [c['mean'] for c in results['C2_n_sweep']['curves']]
sds = [c['std'] for c in results['C2_n_sweep']['curves']]
ax.errorbar(ns, mns, yerr=sds, fmt='o-', capsize=4, label='ensemble (mean ± std)')
ax.axhline(results['C3_oracle_disagreement']['oracle_pool'], color='k', linestyle=':',
            label=f'oracle (full pool)')
ax.set_xlabel('N (ensemble size)'); ax.set_ylabel('test acc')
ax.set_title('C2 — N-sweep'); ax.legend(); ax.grid(True, alpha=0.3)

# (c) C2 std vs theoretical √N
ax = axes[2]
single_std = sds[0]
theo = [single_std / math.sqrt(n) for n in ns]
ax.plot(ns, sds, 'o-', label='observed std')
ax.plot(ns, theo, '--', label='single_std / √N (bagging)')
ax.set_xlabel('N'); ax.set_ylabel('std of ensemble acc')
ax.set_title('C2 — std vs √N reference')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.savefig('summary_kws_v2.png', dpi=100); plt.show()
print('Saved: summary_kws_v2.png, results_kws_v2.json')

## What to read first after running

1. **C1 ensemble gains across arms.** If `hash_only` matches `all_diff` within 0.2% — diploma narrative is intact ("gain comes from hash diversity"). If `init_only` matches them too, the gain comes from training noise, not from hashes. If only `all_diff` wins, hash + init compound.
2. **C2 N=1..6 std curve.** Compare to single_std/√N reference. If observed std falls *faster* than √N, super-bagging is real and worth highlighting. If it tracks √N, the v1 ×3.84 was an artifact.
3. **C3 oracle headroom.** If oracle − ens > 3%, smart routing has room to work. If < 1%, simple `mean_logits` is asymptotically optimal and we can stop investigating routers.
4. **A3 hetero vs all_diff homo.** Replicates v1's negative result on H2 with cleaner control.